In [102]:
import pandas as pd
import numpy as np

In [123]:
df1 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_1.csv')
df2 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_2.csv')
df3 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_3.csv')
players_anomaly = pd.read_csv('../dataset/clean/players_with_anomaly.csv')

df = pd.concat([df1, df2, df3], axis = 0)
def convert_minutes(x):
    if pd.isna(x):
        return None
    try:
        minutes, seconds = map(int, x.split(':'))
        return minutes + seconds / 60
    except:
        return None

df['minutes'] = df['minutes'].apply(convert_minutes)
df = df.fillna(0)
players_anomaly.columns

Index(['gameId', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted',
       'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted',
       'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted',
       'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive',
       'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers',
       'foulsPersonal', 'points', 'plusMinusPoints'],
      dtype='object')

In [125]:
league_ppp = 1.08  # substitua se tiver valor exato
team_pace = 100    # substitua com valor real
league_pace = 100  # substitua com valor real

players_anomaly['offensive_possessions'] = (
    players_anomaly['fieldGoalsAttempted'] +
    0.44 * players_anomaly['freeThrowsAttempted'] +
    players_anomaly['turnovers'] -
    players_anomaly['reboundsOffensive']
)

players_anomaly['points_produced'] = players_anomaly['points'] + 0.5 * players_anomaly['assists']
players_anomaly['marginal_offense'] = (
    players_anomaly['points_produced'] - 0.92 * league_ppp * players_anomaly['offensive_possessions']
)
marginal_points_per_win = 0.32 * 100 * (team_pace / league_pace)
players_anomaly['OWS'] = players_anomaly['marginal_offense'] / marginal_points_per_win
players_anomaly.head(5)

,gameId,minutes,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,threePointersMade,threePointersAttempted,threePointersPercentage,freeThrowsMade,freeThrowsAttempted,...,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints,offensive_possessions,points_produced,marginal_offense,OWS
0,22200212,1.766667,1,1,1.0,0,0,0.0,0,0,...,0,0,0,1,2,-5,1.0,2.0,1.0064,0.03145
1,22200237,0.000000,0,0,0.0,0,0,0.0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0000,0.00000
2,22200317,0.000000,0,0,0.0,0,0,0.0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0000,0.00000
3,22200341,0.000000,0,0,0.0,0,0,0.0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0000,0.00000
4,22200512,0.000000,0,0,0.0,0,0,0.0,0,0,...,0,0,0,0,0,0,0.0,0.0,0.0000,0.00000


In [126]:
players_anomaly['defensive_rating'] = 106  # ou use real por jogador

players_anomaly['team_minutes'] = 5 * 48  # 5 jogadores * 48 min
players_anomaly['team_def_possessions'] = players_anomaly['team_minutes']  # aproximação simplificada

players_anomaly['marginal_defense'] = (
    (players_anomaly['minutes'] / players_anomaly['team_minutes']) *
    players_anomaly['team_def_possessions'] *
    (1.08 * league_ppp - players_anomaly['defensive_rating'] / 100)
)

players_anomaly['marginal_points_per_win'] = 0.32 * league_ppp * 100 * (team_pace / league_pace)

players_anomaly['DWS'] = players_anomaly['marginal_defense'] / players_anomaly['marginal_points_per_win']

In [132]:
players_anomaly['DWS'] = players_anomaly['DWS'].replace(0, np.nan)
players_anomaly = players_anomaly.dropna()
players_anomaly['WS'] = players_anomaly['DWS']+players_anomaly['OWS']

In [133]:
players_anomaly.to_csv('../dataset/clean/players_anomaly_with_winshare.csv')